# ARRL International DX Contest CW 2026 — Propagation Analysis

**Contest**: ARRL International DX Contest — CW  
**Dates**: 2026-02-21 00:00 — 2026-02-22 23:59 UTC (48 hours)  
**Bands**: 160m through 10m (no WARC)  
**Mode**: CW  
**Exchange**: W/VE send power, DX send entity  

**Analysis window**: 2026-02-20 00:00 — 2026-02-23 23:59 UTC (24h before + 24h after)  

**Data sources**:  
- **RBN** (primary) — CW skimmer network captured contest signals directly  
- **PSKR** (secondary) — digital mode observations as propagation proxy  
- **Solar indices** — SFI, SSN, Kp, Ap from GFZ Potsdam  
- **DSCOVR** — solar wind Bz, speed, density from L1  

This analysis is produced by the [IONIS-AI](https://github.com/IONIS-AI) and
[QSO-Graph](https://github.com/qso-graph) projects — 14.3B rows of propagation data,
71 MCP tools, and a trained model applied to contest propagation.

---

**Dataset**: `arrl-dx-cw-2026.sqlite`  
**Download**: [SourceForge](https://sourceforge.net/projects/ionis-ai/files/contests/)  

## Setup

In [ ]:
import sqlite3
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from ionis_jupyter import (
    grid_to_latlon,
    grid_distance,
    grid_bearing,
    solar_elevation,
    classify_path,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Band ID reference
BAND_NAMES = {
    102: "160m", 103: "80m", 104: "60m", 105: "40m", 106: "30m",
    107: "20m", 108: "17m", 109: "15m", 110: "12m", 111: "10m",
}
# Contest bands (no WARC: 60m, 30m, 17m, 12m)
CONTEST_BANDS = [102, 103, 105, 107, 109, 111]
CONTEST_BAND_NAMES = {k: BAND_NAMES[k] for k in CONTEST_BANDS}

# Contest timing
CONTEST_START = datetime(2026, 2, 21, 0, 0, tzinfo=timezone.utc)
CONTEST_END = datetime(2026, 2, 22, 23, 59, tzinfo=timezone.utc)
ANALYSIS_START = datetime(2026, 2, 20, 0, 0, tzinfo=timezone.utc)
ANALYSIS_END = datetime(2026, 2, 23, 23, 59, tzinfo=timezone.utc)

## Load Contest Dataset

The contest dataset is a standalone SQLite file with four tables:
- `rbn_signatures` — CW skimmer observations (primary for CW contests)
- `pskr_signatures` — digital mode observations (propagation proxy)
- `solar_timeline` — solar/geomagnetic conditions over the analysis window
- `contest_info` — contest metadata

In [ ]:
# Find the dataset
# Priority: local data/ directory, then IONIS_DATA_DIR, then SourceForge contests/
import os

DATASET_NAME = "arrl-dx-cw-2026.sqlite"

search_paths = [
    Path("../data") / DATASET_NAME,                                    # Repo data dir
    Path(os.environ.get("IONIS_DATA_DIR", "")) / ".." / "contests" / DATASET_NAME,
    Path.home() / ".ionis-mcp" / "data" / "contests" / DATASET_NAME,  # Default location
]

db_path = None
for p in search_paths:
    if p.exists():
        db_path = p
        break

if db_path is None:
    print(f"Dataset not found: {DATASET_NAME}")
    print(f"Download from: https://sourceforge.net/projects/ionis-ai/files/contests/")
    print(f"Place in one of: {[str(p) for p in search_paths]}")
else:
    print(f"Dataset: {db_path}")
    print(f"Size: {db_path.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# Load all tables
conn = sqlite3.connect(str(db_path))

# Contest metadata
contest_info = pd.read_sql_query("SELECT * FROM contest_info", conn)
display(contest_info)

# RBN signatures (primary — direct CW skimmer observations)
rbn = pd.read_sql_query("SELECT * FROM rbn_signatures", conn)
print(f"\nRBN signatures: {len(rbn):,}")

# PSKR signatures (secondary — digital propagation proxy)
pskr = pd.read_sql_query("SELECT * FROM pskr_signatures", conn)
print(f"PSKR signatures: {len(pskr):,}")

# Solar timeline
solar = pd.read_sql_query("SELECT * FROM solar_timeline", conn)
solar["date"] = pd.to_datetime(solar["date"])
print(f"Solar timeline rows: {len(solar):,}")

conn.close()

---

## Section 1 — Contest Overview

The ARRL International DX Contest is one of the premier HF operating events.
In the CW weekend, stations outside the US/Canada work W/VE stations for
DXCC entity multipliers. The contest runs for 48 hours on all HF bands
(160m through 10m, excluding WARC bands).

For propagation analysis, this is ideal:
- **CW mode** means RBN skimmers capture actual contest signals
- **All HF bands** provides a full propagation profile
- **48 hours** gives us day/night cycles across two complete days
- **Massive participation** generates high spot density globally

In [ ]:
# Summary statistics
print("=" * 60)
print("ARRL DX CW 2026 — Dataset Summary")
print("=" * 60)

for label, df in [("RBN (CW skimmers)", rbn), ("PSKR (digital proxy)", pskr)]:
    print(f"\n--- {label} ---")
    print(f"  Total signatures: {len(df):,}")
    print(f"  Total spot count: {df['spot_count'].sum():,.0f}")
    print(f"  Unique TX grids:  {df['tx_grid_4'].nunique():,}")
    print(f"  Unique RX grids:  {df['rx_grid_4'].nunique():,}")
    print(f"  Unique grid pairs: {len(df.groupby(['tx_grid_4', 'rx_grid_4'])):,}")
    print(f"  Bands active: ", end="")
    for b in sorted(df["band"].unique()):
        name = BAND_NAMES.get(b, str(b))
        count = len(df[df["band"] == b])
        print(f"{name}({count:,}) ", end="")
    print()
    print(f"  Distance range: {df['avg_distance'].min():.0f} — {df['avg_distance'].max():.0f} km")
    print(f"  SNR range: {df['median_snr'].min():.1f} — {df['median_snr'].max():.1f} dB")

---

## Section 2 — Solar Conditions Timeline

Solar conditions during the contest window, extended 24 hours before and after.
Key indices:
- **SFI** (Solar Flux Index) — F-layer ionization, higher = better high-band propagation
- **SSN** (Sunspot Number) — solar activity indicator
- **Kp** — geomagnetic disturbance (0-9 scale, >4 = storm)
- **Ap** — daily geomagnetic activity
- **Bz** — interplanetary magnetic field z-component (negative = storm coupling)

In [ ]:
# Solar conditions summary
print("Solar Conditions Summary")
print("-" * 40)
for col in ["sfi", "ssn", "kp", "ap"]:
    if col in solar.columns:
        vals = solar[col].dropna()
        if len(vals) > 0:
            print(f"  {col.upper():>4}: {vals.min():.1f} — {vals.max():.1f}  (mean: {vals.mean():.1f})")

# Check for DSCOVR data
if "bz" in solar.columns:
    bz = solar["bz"].dropna()
    if len(bz) > 0:
        print(f"    Bz: {bz.min():.1f} — {bz.max():.1f} nT")
        neg_pct = (bz < 0).sum() / len(bz) * 100
        print(f"    Bz negative: {neg_pct:.0f}% of measurements")

In [ ]:
# Solar timeline plots
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# SFI
ax = axes[0]
if "sfi" in solar.columns:
    ax.plot(solar["date"], solar["sfi"], "o-", color="darkorange", markersize=4, label="SFI")
    ax.set_ylabel("SFI (SFU)")
    ax.legend(loc="upper right")
    ax.set_title("Solar Flux Index")

# Kp
ax = axes[1]
if "kp" in solar.columns:
    colors = ["green" if k < 3 else "orange" if k < 5 else "red" for k in solar["kp"]]
    ax.bar(solar["date"], solar["kp"], color=colors, width=0.1, alpha=0.7)
    ax.axhline(4, color="red", linestyle="--", alpha=0.5, label="Storm threshold")
    ax.set_ylabel("Kp Index")
    ax.set_ylim(0, 9)
    ax.legend(loc="upper right")
    ax.set_title("Geomagnetic Activity")

# Bz (if available)
ax = axes[2]
if "bz" in solar.columns and solar["bz"].notna().any():
    ax.plot(solar["date"], solar["bz"], ".-", color="steelblue", markersize=3)
    ax.axhline(0, color="gray", linestyle="-", alpha=0.5)
    ax.fill_between(solar["date"], solar["bz"], 0,
                    where=(solar["bz"] < 0), color="red", alpha=0.15, label="Southward (storm coupling)")
    ax.set_ylabel("Bz (nT)")
    ax.legend(loc="upper right")
    ax.set_title("Interplanetary Magnetic Field (DSCOVR L1)")
else:
    ax.text(0.5, 0.5, "DSCOVR Bz data not available", transform=ax.transAxes, ha="center")
    ax.set_title("Interplanetary Magnetic Field")

# Contest window shading
for ax in axes:
    ax.axvspan(CONTEST_START, CONTEST_END, alpha=0.08, color="blue", label="Contest window")
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate(rotation=30)
fig.suptitle("Solar Conditions — ARRL DX CW 2026 Analysis Window", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 3 — Band Activity Heatmap

Hour (UTC) x Band matrix showing spot density. This reveals:
- Which bands were open at which times
- Peak activity periods
- The contest "light switch" effect — activity erupts at 00:00 UTC Saturday
  and drops off sharply at 23:59 UTC Sunday

In [ ]:
def plot_activity_heatmap(df, title, ax=None):
    """Plot hour x band activity heatmap."""
    # Filter to contest bands
    contest_df = df[df["band"].isin(CONTEST_BANDS)]

    # Pivot: hour (rows) x band (columns), sum spot_count
    pivot = contest_df.groupby(["hour", "band"])["spot_count"].sum().unstack(fill_value=0)

    # Rename columns to band names
    pivot.columns = [BAND_NAMES.get(b, str(b)) for b in pivot.columns]

    # Reorder columns by frequency (160m → 10m)
    band_order = [BAND_NAMES[b] for b in CONTEST_BANDS if BAND_NAMES[b] in pivot.columns]
    pivot = pivot[band_order]

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        pivot,
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt=",.0f",
        cbar_kws={"label": "Total Spot Count"},
        linewidths=0.5,
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Band")
    ax.set_ylabel("Hour (UTC)")
    return ax

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

plot_activity_heatmap(rbn, "RBN — CW Skimmer Activity", ax=axes[0])
plot_activity_heatmap(pskr, "PSKR — Digital Mode Activity", ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Band-by-band activity timeline (RBN — primary source)
fig, axes = plt.subplots(len(CONTEST_BANDS), 1, figsize=(14, 3 * len(CONTEST_BANDS)), sharex=True)

for i, band_id in enumerate(CONTEST_BANDS):
    ax = axes[i]
    band_df = rbn[rbn["band"] == band_id]
    hour_activity = band_df.groupby("hour")["spot_count"].sum()

    ax.bar(hour_activity.index, hour_activity.values, color="steelblue", alpha=0.7)
    ax.set_ylabel("Spots")
    ax.set_title(f"{BAND_NAMES[band_id]} — RBN Activity by Hour", fontsize=11)
    ax.set_xlim(-0.5, 23.5)
    ax.grid(True, alpha=0.3, axis="y")

axes[-1].set_xlabel("Hour (UTC)")
axes[-1].set_xticks(range(0, 24))
fig.suptitle("Band-by-Band Activity — RBN CW Skimmers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 4 — Geographic Reach

Per-band analysis of geographic coverage:
- Distance distribution histograms
- Continental breakdown (NA-EU, NA-AS, NA-SA, etc.)
- Top grid pairs by spot density

In [ ]:
# Distance distribution per band
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, band_id in enumerate(CONTEST_BANDS):
    ax = axes[i // 3][i % 3]
    band_df = rbn[rbn["band"] == band_id]

    if len(band_df) > 0:
        ax.hist(band_df["avg_distance"], bins=50, color="steelblue", alpha=0.7,
                weights=band_df["spot_count"])
        median_dist = np.average(band_df["avg_distance"], weights=band_df["spot_count"])
        ax.axvline(median_dist, color="red", linestyle="--", alpha=0.7,
                   label=f"Weighted mean: {median_dist:,.0f} km")
        ax.legend(fontsize=9)

    ax.set_title(f"{BAND_NAMES[band_id]}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Spot-Weighted Count")

fig.suptitle("Distance Distribution by Band — RBN", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Continental breakdown using grid prefixes
# Simplified continent mapping from grid field (first 2 chars)
def grid_to_continent(grid):
    """Rough continent classification from Maidenhead grid field."""
    if not grid or len(grid) < 2:
        return "Unknown"
    field = grid[:2].upper()
    lon_field = ord(field[0]) - ord('A')  # 0-17
    lat_field = ord(field[1]) - ord('A')  # 0-17

    # Approximate continent boundaries
    if lat_field >= 10 and 7 <= lon_field <= 13:  # Northern Europe
        return "EU"
    if lat_field >= 7 and 9 <= lon_field <= 13:   # Europe/Mediterranean
        return "EU"
    if lat_field >= 5 and 3 <= lon_field <= 7:    # North America
        return "NA"
    if lat_field >= 2 and 4 <= lon_field <= 7:    # Central/South America
        return "SA"
    if lat_field >= 5 and 13 <= lon_field <= 17:  # Asia
        return "AS"
    if lat_field <= 5 and 13 <= lon_field <= 16:  # Oceania
        return "OC"
    if lat_field <= 6 and 9 <= lon_field <= 13:   # Africa
        return "AF"
    return "Other"


# Add continent classification
rbn_geo = rbn.copy()
rbn_geo["tx_continent"] = rbn_geo["tx_grid_4"].apply(grid_to_continent)
rbn_geo["rx_continent"] = rbn_geo["rx_grid_4"].apply(grid_to_continent)
rbn_geo["path"] = rbn_geo["tx_continent"] + "-" + rbn_geo["rx_continent"]

# Top intercontinental paths
path_stats = (
    rbn_geo.groupby("path")
    .agg(signatures=("spot_count", "count"), total_spots=("spot_count", "sum"))
    .sort_values("total_spots", ascending=False)
    .head(15)
)

print("Top Intercontinental Paths (RBN)")
print("=" * 45)
for path, row in path_stats.iterrows():
    print(f"  {path:>10}: {row['signatures']:>8,} sigs, {row['total_spots']:>10,} spots")

In [ ]:
# Intercontinental path breakdown by band
top_paths = path_stats.index[:8].tolist()
inter_df = rbn_geo[rbn_geo["path"].isin(top_paths) & rbn_geo["band"].isin(CONTEST_BANDS)]

fig, ax = plt.subplots(figsize=(14, 7))

pivot = inter_df.groupby(["path", "band"])["spot_count"].sum().unstack(fill_value=0)
pivot.columns = [BAND_NAMES.get(b, str(b)) for b in pivot.columns]
band_order = [BAND_NAMES[b] for b in CONTEST_BANDS if BAND_NAMES[b] in pivot.columns]
pivot = pivot[band_order]

pivot.plot(kind="barh", stacked=True, ax=ax, colormap="tab10")
ax.set_xlabel("Total Spots")
ax.set_ylabel("Path")
ax.set_title("Intercontinental Paths by Band — RBN", fontsize=13, fontweight="bold")
ax.legend(title="Band", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---

## Section 5 — Day/Night Analysis

Solar terminator classification of propagation paths:
- **Both-day**: TX and RX both in sunlight
- **Cross-terminator**: one end in daylight, one in darkness (greyline paths)
- **Both-dark**: TX and RX both in darkness

This is where the IONIS model's darkness cross-products (V22-gamma) are validated.
Low bands (160m, 80m) should dominate both-dark paths. High bands (10m, 15m)
should be absent or very weak in both-dark conditions.

In [ ]:
# Classify each signature by solar geometry
# Use the midpoint of each hour for solar elevation

def classify_signature(row):
    """Classify a signature row by solar geometry using contest date."""
    # Use Feb 21 (first contest day) as reference
    dt = datetime(2026, 2, 21, int(row["hour"]), 30, tzinfo=timezone.utc)
    return classify_path(row["tx_grid_4"], row["rx_grid_4"], dt)

# Apply to RBN data (sample if too large for performance)
rbn_class = rbn[rbn["band"].isin(CONTEST_BANDS)].copy()
if len(rbn_class) > 100_000:
    print(f"Classifying {len(rbn_class):,} signatures (may take a moment)...")

rbn_class["solar_class"] = rbn_class.apply(classify_signature, axis=1)

# Summary
class_counts = rbn_class.groupby("solar_class")["spot_count"].sum()
total = class_counts.sum()
print("\nSolar Geometry Classification (RBN)")
print("=" * 40)
for cls, count in class_counts.items():
    print(f"  {cls:>20}: {count:>10,} spots ({count/total*100:.1f}%)")

In [ ]:
# Day/night breakdown by band
fig, ax = plt.subplots(figsize=(12, 7))

class_by_band = (
    rbn_class.groupby(["band", "solar_class"])["spot_count"]
    .sum()
    .unstack(fill_value=0)
)
class_by_band.index = [BAND_NAMES.get(b, str(b)) for b in class_by_band.index]

# Normalize to percentages per band
class_pct = class_by_band.div(class_by_band.sum(axis=1), axis=0) * 100

colors = {"both-day": "#FFD700", "cross-terminator": "#4682B4", "both-dark": "#2F4F4F"}
col_order = [c for c in ["both-day", "cross-terminator", "both-dark"] if c in class_pct.columns]
class_pct[col_order].plot(
    kind="bar", stacked=True, ax=ax,
    color=[colors[c] for c in col_order],
)

ax.set_ylabel("Percentage of Spots")
ax.set_xlabel("Band")
ax.set_title("Solar Geometry by Band — RBN", fontsize=13, fontweight="bold")
ax.legend(title="Path Type")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# SNR by solar classification and band
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, band_id in enumerate(CONTEST_BANDS):
    ax = axes[i // 3][i % 3]
    band_df = rbn_class[rbn_class["band"] == band_id]

    if len(band_df) > 0:
        for cls in ["both-day", "cross-terminator", "both-dark"]:
            cls_df = band_df[band_df["solar_class"] == cls]
            if len(cls_df) > 0:
                ax.hist(cls_df["median_snr"], bins=30, alpha=0.5,
                        label=f"{cls} ({len(cls_df):,})",
                        color=colors.get(cls, "gray"))

    ax.set_title(BAND_NAMES[band_id], fontsize=12, fontweight="bold")
    ax.set_xlabel("Median SNR (dB)")
    ax.legend(fontsize=8)

fig.suptitle("SNR Distribution by Solar Geometry — RBN", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 6 — SNR Distribution

Signal strength analysis across bands, distances, and time-of-day.

In [ ]:
# SNR box plots by band
fig, ax = plt.subplots(figsize=(12, 6))

band_data = []
band_labels = []
for band_id in CONTEST_BANDS:
    data = rbn[rbn["band"] == band_id]["median_snr"].dropna()
    if len(data) > 0:
        band_data.append(data.values)
        band_labels.append(BAND_NAMES[band_id])

bp = ax.boxplot(band_data, labels=band_labels, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("steelblue")
    patch.set_alpha(0.6)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Band")
ax.set_ylabel("Median SNR (dB)")
ax.set_title("SNR Distribution by Band — RBN", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# SNR vs Distance scatter (per band)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, band_id in enumerate(CONTEST_BANDS):
    ax = axes[i // 3][i % 3]
    band_df = rbn[rbn["band"] == band_id]

    if len(band_df) > 0:
        # Use spot_count for marker size
        sizes = np.clip(band_df["spot_count"].values, 1, 100)
        scatter = ax.scatter(
            band_df["avg_distance"], band_df["median_snr"],
            s=sizes, alpha=0.3, c="steelblue", edgecolors="none",
        )
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

    ax.set_title(BAND_NAMES[band_id], fontsize=12, fontweight="bold")
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Median SNR (dB)")

fig.suptitle("SNR vs Distance — RBN", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# SNR by hour (box plots) — top 3 bands by activity
top_bands = rbn[rbn["band"].isin(CONTEST_BANDS)].groupby("band")["spot_count"].sum().nlargest(3).index

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, band_id in enumerate(top_bands):
    ax = axes[i]
    band_df = rbn[rbn["band"] == band_id]

    # Group by hour for box plots
    hour_groups = [band_df[band_df["hour"] == h]["median_snr"].values
                   for h in range(24)]
    hour_groups = [g if len(g) > 0 else [np.nan] for g in hour_groups]

    bp = ax.boxplot(hour_groups, labels=range(24), patch_artist=True)
    for patch in bp["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.5)

    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("Median SNR (dB)")
    ax.set_title(f"{BAND_NAMES[band_id]} — SNR by Hour", fontsize=12, fontweight="bold")

fig.suptitle("Hourly SNR Distribution — Top 3 Bands (RBN)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Section 7 — IONIS V22-gamma Comparison

Generate IONIS model predictions for observed RBN paths and compare with
measured SNR. This validates the model against independent CW contest data
(not in the training set).

*This section requires the V22-gamma checkpoint and `ionis-training` package.
Skip if not available.*

In [ ]:
# Placeholder — model comparison requires V22-gamma checkpoint
# TODO: Load checkpoint, generate predictions for observed paths,
#       scatter plot predicted vs observed SNR, residual analysis

print("Section 7 — IONIS model comparison")
print("Requires: ionis-training + V22-gamma checkpoint")
print("Status: Pending — will be filled in when model inference is integrated")

---

## Section 8 — Historical Context

Compare this contest weekend with historical CW contest propagation.
The IONIS `contest.signatures` dataset contains 5.7M signatures from
CQ contests (2005-2025). While not ARRL DX specifically, the band/hour
patterns during February CW contests provide a useful baseline.

In [ ]:
# Load historical contest signatures for comparison
try:
    from ionis_jupyter import load_dataset
    contest_hist = load_dataset("contest")
    has_historical = True
    print(f"Loaded {len(contest_hist):,} historical contest signatures")
except (FileNotFoundError, Exception) as e:
    has_historical = False
    print(f"Historical contest data not available: {e}")
    print("Install with: ionis-download --datasets contest")

In [ ]:
if has_historical:
    # Filter historical data to February (same month as ARRL DX CW)
    hist_feb = contest_hist[contest_hist["month"] == 2]
    print(f"Historical February contest signatures: {len(hist_feb):,}")

    # Compare band activity: historical February vs this contest
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Historical February
    ax = axes[0]
    hist_bands = hist_feb[hist_feb["band"].isin(CONTEST_BANDS)]
    hist_pivot = hist_bands.groupby(["hour", "band"])["spot_count"].sum().unstack(fill_value=0)
    hist_pivot.columns = [BAND_NAMES.get(b, str(b)) for b in hist_pivot.columns]
    band_order = [BAND_NAMES[b] for b in CONTEST_BANDS if BAND_NAMES[b] in hist_pivot.columns]
    if band_order:
        hist_pivot[band_order].plot(ax=ax)
    ax.set_title("Historical February Contests (2005-2025)", fontsize=12)
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("Total Spots")
    ax.legend(title="Band")

    # This contest (RBN)
    ax = axes[1]
    rbn_bands = rbn[rbn["band"].isin(CONTEST_BANDS)]
    rbn_pivot = rbn_bands.groupby(["hour", "band"])["spot_count"].sum().unstack(fill_value=0)
    rbn_pivot.columns = [BAND_NAMES.get(b, str(b)) for b in rbn_pivot.columns]
    band_order = [BAND_NAMES[b] for b in CONTEST_BANDS if BAND_NAMES[b] in rbn_pivot.columns]
    if band_order:
        rbn_pivot[band_order].plot(ax=ax)
    ax.set_title("ARRL DX CW 2026 (RBN)", fontsize=12)
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("Total Spots")
    ax.legend(title="Band")

    fig.suptitle("Band Activity: Historical vs Current", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping historical comparison — contest dataset not available.")

In [ ]:
if has_historical:
    # Solar cycle context
    # SFI during this contest vs historical February averages
    try:
        solar_hist = load_dataset("solar")
        solar_hist["date"] = pd.to_datetime(solar_hist["date"])
        solar_feb = solar_hist[solar_hist["date"].dt.month == 2]

        # Average SFI by year for February
        solar_feb_yearly = solar_feb.groupby(solar_feb["date"].dt.year)["sfi"].mean()

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(solar_feb_yearly.index, solar_feb_yearly.values, color="steelblue", alpha=0.6)

        # Highlight this year
        current_sfi = solar["sfi"].mean() if "sfi" in solar.columns else None
        if current_sfi and 2026 in solar_feb_yearly.index:
            ax.bar(2026, solar_feb_yearly[2026], color="darkorange", alpha=0.8, label=f"2026: SFI {current_sfi:.0f}")
        elif current_sfi:
            ax.bar(2026, current_sfi, color="darkorange", alpha=0.8, label=f"2026: SFI {current_sfi:.0f}")

        ax.set_xlabel("Year")
        ax.set_ylabel("Average February SFI")
        ax.set_title("Solar Flux Index — February Average by Year", fontsize=13, fontweight="bold")
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Solar historical comparison skipped: {e}")

---

## Section 9 — Key Findings

Summary of propagation highlights from the ARRL DX CW 2026 contest weekend.

In [ ]:
# Auto-generate summary statistics for the findings section
print("=" * 60)
print("KEY FINDINGS — ARRL DX CW 2026")
print("=" * 60)

# Solar conditions
if "sfi" in solar.columns:
    sfi_mean = solar["sfi"].mean()
    print(f"\n1. Solar conditions: SFI {sfi_mean:.0f} (moderate)")

if "kp" in solar.columns:
    kp_max = solar["kp"].max()
    kp_mean = solar["kp"].mean()
    if kp_max >= 4:
        print(f"   Geomagnetic storm: Kp reached {kp_max:.1f} (mean {kp_mean:.1f})")
    else:
        print(f"   Geomagnetic: quiet to unsettled (Kp max {kp_max:.1f}, mean {kp_mean:.1f})")

# Band activity
print(f"\n2. RBN captured {rbn['spot_count'].sum():,.0f} total spots across {len(rbn):,} signatures")
print(f"   PSKR captured {pskr['spot_count'].sum():,.0f} total spots across {len(pskr):,} signatures")

# Most active band
band_totals = rbn[rbn["band"].isin(CONTEST_BANDS)].groupby("band")["spot_count"].sum().sort_values(ascending=False)
if len(band_totals) > 0:
    top_band = band_totals.index[0]
    print(f"\n3. Most active band: {BAND_NAMES[top_band]} ({band_totals.iloc[0]:,.0f} spots)")
    if len(band_totals) > 1:
        print(f"   Runner-up: {BAND_NAMES[band_totals.index[1]]} ({band_totals.iloc[1]:,.0f} spots)")

# Geographic reach
max_dist = rbn["avg_distance"].max()
mean_dist = np.average(rbn["avg_distance"], weights=rbn["spot_count"])
print(f"\n4. Geographic reach: max {max_dist:,.0f} km, weighted mean {mean_dist:,.0f} km")
print(f"   Unique TX grids: {rbn['tx_grid_4'].nunique():,}, RX grids: {rbn['rx_grid_4'].nunique():,}")

# Day/night
if "solar_class" in rbn_class.columns:
    dark_spots = rbn_class[rbn_class["solar_class"] == "both-dark"]["spot_count"].sum()
    total_spots = rbn_class["spot_count"].sum()
    print(f"\n5. Both-dark paths: {dark_spots:,.0f} spots ({dark_spots/total_spots*100:.1f}% of total)")

print("\n" + "=" * 60)
print("Analysis produced by IONIS-AI + QSO-Graph")
print(f"Dataset: arrl-dx-cw-2026.sqlite")
print(f"Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

---

*Analysis by [IONIS-AI](https://github.com/IONIS-AI) and [QSO-Graph](https://github.com/qso-graph).  
Data: 14.3B propagation observations, 71 MCP tools.  
License: MIT*